In [1]:
import pandas as pd

In [2]:
PRED_DF = "../predict_output/IJF_Docker_predict_result/org_pairs_canonical_predict.csv"
GT_DF = "../dataset/org_pairs/org_pairs.csv"

In [6]:
preds_df = pd.read_csv(PRED_DF)
gt_df = pd.read_csv(GT_DF)

In [7]:
name_to_canonical = dict(
    zip(gt_df["cleaned"], gt_df["canonical_id"])
)

In [ ]:
preds_df = preds_df[["name1", "name2", "score", "pred"]].drop_duplicates()
preds_df["canon1"] = preds_df["name1"].map(name_to_canonical)
preds_df["canon2"] = preds_df["name2"].map(name_to_canonical)
preds_df["correct"] = preds_df["match"] == preds_df["pred"]

In [9]:
# remove self-matches
preds_df.drop(preds_df[preds_df["name1"] == preds_df["name2"]].index, inplace=True)
# remove directional duplicates
preds_df["sorted_names"] = preds_df.apply(lambda row: tuple(sorted([row["name1"], row["name2"]])), axis=1)
preds_df.drop_duplicates(subset="sorted_names", inplace=True)
preds_df.drop(columns="sorted_names", inplace=True)

In [10]:
accuracy = preds_df["correct"].mean()
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9874


In [11]:
precision = (preds_df["pred"] & preds_df["match"]).sum() / preds_df["pred"].sum()
recall = (preds_df["pred"] & preds_df["match"]).sum() / preds_df["match"].sum()
f1 = 2 * (precision * recall) / (precision + recall)
print(f"F1 Score: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

F1 Score: 0.2373
Precision: 0.1643
Recall: 0.4265


In [12]:
preds_df["canon1"] = preds_df["name1"].map(name_to_canonical)
preds_df["canon2"] = preds_df["name2"].map(name_to_canonical)
preds_df["is_true_pos"] = preds_df["canon1"] == preds_df["canon2"]

print(f"True positives in top k: {preds_df['is_true_pos'].sum()}. This is the number of pairs in the top k that are actually the same organization.")

True positives in top k: 748. This is the number of pairs in the top k that are actually the same organization.


In [13]:
num_canonical = len(pd.concat([preds_df["canon1"], preds_df["canon2"]]).unique())
canon_ids = pd.concat([preds_df["canon1"], preds_df["canon2"]]).unique()
print(f"Number of unique canonical organizations: {num_canonical}. This is the number of entities in the dataset.")

Number of unique canonical organizations: 322. This is the number of entities in the dataset.


In [14]:
num_names = preds_df[["name1", "name2"]].nunique().sum()
names = pd.concat([preds_df["name1"], preds_df["name2"]]).unique()
print(f"Number of unique names: {num_names}. This is the number of unique organization names in the dataset.")

Number of unique names: 1165. This is the number of unique organization names in the dataset.


In [15]:
preds_df["canon1"] = preds_df["name1"].map(name_to_canonical)
preds_df["canon2"] = preds_df["name2"].map(name_to_canonical)
preds_df["is_true_pos"] = preds_df["canon1"] == preds_df["canon2"]

print(f"True positives in top k: {preds_df['is_true_pos'].sum()}. This is the number of pairs in the top k that are actually the same organization.")

total_theoretical_pairs = 0
for id in canon_ids:
    num_names = [name_to_canonical.get(name) == id for name in names].count(True)
    theoretical_num_pairs = num_names * (num_names - 1) // 2
    total_theoretical_pairs += theoretical_num_pairs
print(f"Total theoretical pairs: {total_theoretical_pairs}. This is the total number of pairs that could be formed if we had all possible pairs of names for each canonical organization.")

True positives in top k: 748. This is the number of pairs in the top k that are actually the same organization.
Total theoretical pairs: 749. This is the total number of pairs that could be formed if we had all possible pairs of names for each canonical organization.


In [16]:
preds_df[preds_df["is_true_pos"]]

,name1,name2,score,pred,canon1,canon2,match,correct,is_true_pos
0,ADIRONDACK IN JOINT VENTURE WITH AMITA ARTEMP ...,ADIRONDACK INFORMATION MANAGEMENT AMITA ARTEMP...,0.925886,1,188,188,True,True,True
1,ADIRONDACK IN JOINT VENTURE WITH AMITA ARTEMP ...,ADIRONDACK INFORMATION MANAGEMENT AIM GROUP IN...,0.815210,1,188,188,True,True,True
2,ADIRONDACK IN JOINT VENTURE WITH AMITA ARTEMP ...,ADIRONDACK INFORMATION MANAGEMENT AMITA ARTEM ...,0.925709,1,188,188,True,True,True
3,ADIRONDACK IN JOINT VENTURE WITH AMITA ARTEMP ...,ADIRONDACK INFORMATION MANAGEMENT AMITA ARTEMP...,0.936164,1,188,188,True,True,True
4,ADIRONDACK IN JOINT VENTURE WITH AMITA ARTEMP ...,ADIRONDACK INFORMATION MANGEMENT INV AMITA ART...,0.845812,1,188,188,True,True,True
...,...,...,...,...,...,...,...,...,...
261127,ALTIS HUMAN RESOURCES OTTAWA EXCEL HUMAN RESOU...,EXCELHR,0.369320,0,26,26,True,False,True
261343,ALTIS HUMAN RESOURCES OTTAWA EXCEL HUMAN RESOU...,EXCEL ITR,0.266119,0,26,26,True,False,True
280955,ITNETOTTAWA,KPMGVAN,0.002471,0,86,86,True,False,True
281438,KPMG IN JOINT VENTURE,KPMGVAN,0.864790,1,86,86,True,True,True


In [77]:
preds_df[preds_df["pred"] == 1]

,name1,name2,score,pred,canon1,canon2,match,correct,is_true_pos
0,ADIRONDACK IN JOINT VENTURE WITH AMITA ARTEMP ...,ADIRONDACK INFORMATION MANAGEMENT AMITA ARTEMP...,0.925886,1,188,188,True,True,True
1,ADIRONDACK IN JOINT VENTURE WITH AMITA ARTEMP ...,ADIRONDACK INFORMATION MANAGEMENT AIM GROUP IN...,0.815210,1,188,188,True,True,True
2,ADIRONDACK IN JOINT VENTURE WITH AMITA ARTEMP ...,ADIRONDACK INFORMATION MANAGEMENT AMITA ARTEM ...,0.925709,1,188,188,True,True,True
3,ADIRONDACK IN JOINT VENTURE WITH AMITA ARTEMP ...,ADIRONDACK INFORMATION MANAGEMENT AMITA ARTEMP...,0.936164,1,188,188,True,True,True
4,ADIRONDACK IN JOINT VENTURE WITH AMITA ARTEMP ...,ADIRONDACK INFORMATION MANGEMENT INV AMITA ART...,0.845812,1,188,188,True,True,True
...,...,...,...,...,...,...,...,...,...
261472,ALTIS HUMAN RESOURCES OTTAWA EXCEL HUMAN RESOU...,ITEX,0.983777,1,26,506,False,False,False
261473,ALTIS HUMAN RESOURCES OTTAWA EXCEL HUMAN RESOU...,DEW ENGINEERING DEVELOPMENT,0.956269,1,26,350,False,False,False
261474,ALTIS HUMAN RESOURCES OTTAWA EXCEL HUMAN RESOU...,TKI CONSTRUCTION,0.979742,1,26,765,False,False,False
261475,ALTIS HUMAN RESOURCES OTTAWA EXCEL HUMAN RESOU...,BLACK MCDONALD,0.964063,1,26,107,False,False,False


In [96]:
preds_df[(preds_df["is_true_pos"]) & (preds_df["pred"] != 1)]

,name1,name2,score,pred,canon1,canon2,match,correct,is_true_pos
1250,NO SUPPLIER NAMED,GENERIC PSPCMANAGE SUPPLIER ACCOUNT,0.002119,0,62,62,True,False,True
1513,EXCELHR,ALTIS HR,0.005662,0,26,26,True,False,True
1718,EXCELHR,ALTIS RECRUITMENT,0.004496,0,26,26,True,False,True
1724,EXCELHR,ALTIS HUMAN RESOURCES,0.027909,0,26,26,True,False,True
1741,EXCELHR,ALTIS PROFESSIONAL RECRUITMENT,0.005132,0,26,26,True,False,True
...,...,...,...,...,...,...,...,...,...
250625,INNOVATIVE DRILLING,ROYAL ENVIRONMENTAL,0.001813,0,348,348,True,False,True
261127,ALTIS HUMAN RESOURCES OTTAWA EXCEL HUMAN RESOU...,EXCELHR,0.369320,0,26,26,True,False,True
261343,ALTIS HUMAN RESOURCES OTTAWA EXCEL HUMAN RESOU...,EXCEL ITR,0.266119,0,26,26,True,False,True
280955,ITNETOTTAWA,KPMGVAN,0.002471,0,86,86,True,False,True
